In [2]:
!pip install -q datasets transformers[torch]

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import requests

# Step 1: Load Shakespeare's sonnets dataset
url = "https://raw.githubusercontent.com/quadrismegistus/prosodic/refs/heads/master/corpora/corppoetry_en/en.shakespeare.txt"
response = requests.get(url)
shakespeare_text = response.text

# Step 2: Prepare the dataset (split by line)
def prepare_data(text):
    lines = text.splitlines()
    # Filter out empty lines
    examples = [line for line in lines if line.strip()]
    return Dataset.from_dict({"text": examples})

# Create dataset from text split by line
dataset = prepare_data(shakespeare_text)
# Step 3: Load GPT-2 model and tokenizer
model_name = "gpt2"  # or use "gpt2-medium" or "gpt2-large" for larger versions
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add padding token to the tokenizer
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Step 4: Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=500,
    evaluation_strategy="steps",
    eval_steps=500,
    weight_decay=0.01,
    save_total_limit=2
)

# Step 5: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset
)

# Step 6: Start training
trainer.train()

# Step 7: Save the model and tokenizer
model.save_pretrained("./fine_tuned_gpt2_shakespeare")
tokenizer.save_pretrained("./fine_tuned_gpt2_shakespeare")

print("Fine-tuning complete and model saved!")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip


Map:   0%|          | 0/2155 [00:00<?, ? examples/s]

/Users/ryan/github/llmdh/venv/lib/python3.12/site-packages/accelerate/accelerator.py:457: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


  0%|          | 0/1617 [00:00<?, ?it/s]

...

{'eval_loss': 0.35461270809173584, 'eval_runtime': 41.2869, 'eval_samples_per_second': 52.196, 'eval_steps_per_second': 13.055, 'epoch': 0.93}
{'loss': 0.4038, 'grad_norm': 1.7372357845306396, 'learning_rate': 4.955237242614145e-05, 'epoch': 0.95}
{'loss': 0.3905, 'grad_norm': 1.7091615200042725, 'learning_rate': 4.91047448522829e-05, 'epoch': 0.96}
{'loss': 0.3827, 'grad_norm': 1.6008142232894897, 'learning_rate': 4.865711727842435e-05, 'epoch': 0.98}
{'loss': 0.3802, 'grad_norm': 1.4056404829025269, 'learning_rate': 4.82094897045658e-05, 'epoch': 1.0}
{'loss': 0.3793, 'grad_norm': 1.544768214225769, 'learning_rate': 4.776186213070725e-05, 'epoch': 1.02}
{'loss': 0.3844, 'grad_norm': 1.702734351158142, 'learning_rate': 4.7314234556848704e-05, 'epoch': 1.04}
{'loss': 0.3592, 'grad_norm': 1.9841424226760864, 'learning_rate': 4.686660698299015e-05, 'epoch': 1.06}
{'loss': 0.3625, 'grad_norm': 1.6942323446273804, 'learning_rate': 4.6418979409131604e-05, 'epoch': 1.08}
{'loss': 0.3611, 'gr

  0%|          | 0/539 [00:00<?, ?it/s]

{'eval_loss': 0.3066357672214508, 'eval_runtime': 43.1982, 'eval_samples_per_second': 49.886, 'eval_steps_per_second': 12.477, 'epoch': 1.86}
{'loss': 0.3658, 'grad_norm': 1.7845239639282227, 'learning_rate': 2.7170993733213968e-05, 'epoch': 1.87}
{'loss': 0.3521, 'grad_norm': 1.6440341472625732, 'learning_rate': 2.6723366159355418e-05, 'epoch': 1.89}
{'loss': 0.3561, 'grad_norm': 2.0047967433929443, 'learning_rate': 2.6275738585496868e-05, 'epoch': 1.91}
{'loss': 0.3484, 'grad_norm': 2.129312753677368, 'learning_rate': 2.582811101163832e-05, 'epoch': 1.93}
{'loss': 0.3665, 'grad_norm': 1.7578575611114502, 'learning_rate': 2.538048343777977e-05, 'epoch': 1.95}
{'loss': 0.3658, 'grad_norm': 1.828436017036438, 'learning_rate': 2.493285586392122e-05, 'epoch': 1.97}
{'loss': 0.365, 'grad_norm': 1.866133213043213, 'learning_rate': 2.448522829006267e-05, 'epoch': 1.99}
{'loss': 0.3306, 'grad_norm': 1.5434553623199463, 'learning_rate': 2.403760071620412e-05, 'epoch': 2.0}
{'loss': 0.3058, 'gr

  0%|          | 0/539 [00:00<?, ?it/s]

{'eval_loss': 0.27930429577827454, 'eval_runtime': 39.4815, 'eval_samples_per_second': 54.583, 'eval_steps_per_second': 13.652, 'epoch': 2.78}
{'loss': 0.337, 'grad_norm': 1.9061473608016968, 'learning_rate': 4.789615040286482e-06, 'epoch': 2.8}
{'loss': 0.3115, 'grad_norm': 1.8368457555770874, 'learning_rate': 4.341987466427932e-06, 'epoch': 2.82}
{'loss': 0.3075, 'grad_norm': 1.7906627655029297, 'learning_rate': 3.8943598925693825e-06, 'epoch': 2.84}
{'loss': 0.3123, 'grad_norm': 1.5317009687423706, 'learning_rate': 3.4467323187108323e-06, 'epoch': 2.86}
{'loss': 0.3011, 'grad_norm': 2.0501718521118164, 'learning_rate': 2.999104744852283e-06, 'epoch': 2.88}
{'loss': 0.3125, 'grad_norm': 1.661927580833435, 'learning_rate': 2.5514771709937336e-06, 'epoch': 2.89}
{'loss': 0.2941, 'grad_norm': 1.9424148797988892, 'learning_rate': 2.103849597135184e-06, 'epoch': 2.91}
{'loss': 0.3076, 'grad_norm': 1.8521170616149902, 'learning_rate': 1.656222023276634e-06, 'epoch': 2.93}
{'loss': 0.3158, 